In [1]:
import os
import pandas as pd

# Paper 的 50 个合约
PAPER_50 = [
    # Commodities
    "CC","DA","GI","JO","KC","KW","LB","NR","SB",
    "ZA","ZC","ZF","ZG","ZH","ZI","ZK","ZL","ZN",
    "ZO","ZP","ZR","ZT","ZU","ZW","ZZ",
    # Equity Index
    "CA","EN","ER","ES","LX","MD","SC","SP","XU","XX","YM",
    # Fixed Income
    "DT","FB","TY","UB","US",
    # FX
    "AN","BN","CN","DX","FN","JN","MP","NK","SN"
]
# -----------------------------
# 1️⃣ 找 CLC 文件夹
# -----------------------------
def find_clcdata(root_path):
    for root, dirs, files in os.walk(root_path):
        if os.path.basename(root).upper() == "CLC":
            return root
    raise FileNotFoundError("CLC folder not found")
# -----------------------------
# 2️⃣ 读取单个 RAD 文件
# -----------------------------
def read_rad_csv(file_path):
    df = pd.read_csv(file_path, header=None)
    # 按 manual 强制赋列名
    df.columns = [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "open_interest"
    ]
    # 转时间
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    # 排序 + 去重
    df = df.sort_values("date").drop_duplicates("date")

    return df
# -----------------------------
# Main function
# -----------------------------
def load_paper_rad_data(root_path, start=None, end=None):

    clc_path = find_clcdata(root_path)

    data_dict = {}
    missing = []

    # 建立 filename → path 映射（大小写无关）
    file_map = {}
    for root, _, files in os.walk(clc_path):
        for f in files:
            file_map[f.upper()] = os.path.join(root, f)

    for ticker in PAPER_50:
        fname = f"{ticker}_RAD.CSV".upper()

        if fname not in file_map:
            missing.append(ticker)
            continue

        df = read_rad_csv(file_map[fname])

        if start:
            df = df[df["date"] >= pd.to_datetime(start)]
        if end:
            df = df[df["date"] <= pd.to_datetime(end)]

        df["ticker"] = ticker

        data_dict[ticker] = df.reset_index(drop=True)

    # 拼 panel
    panel = pd.concat(data_dict.values(), ignore_index=True)

    return data_dict, panel, missing

In [2]:
import numpy as np
import pandas as pd


def build_paper_features(data_dict, dropna=False):
    """
    Build paper-style features from load_paper_rad_data output.

    Parameters
    ----------
    data_dict : dict
        {ticker: DataFrame}
        Each DataFrame should contain at least:
        ['date', 'open', 'high', 'low', 'close', 'volume', 'open_interest']
        and optionally ['ticker'].

    dropna : bool
        If True, drop rows with missing core feature values.

    Returns
    -------
    feature_dict : dict
        {ticker: feature_df}
    feature_panel : DataFrame
        concatenated long-format feature panel
    """

    feature_dict = {}

    for ticker, df in data_dict.items():
        x = df.copy()

        # -------------------------
        # 1) 基础清洗
        # -------------------------
        required_cols = ["date", "open", "high", "low", "close", "volume", "open_interest"]
        missing_cols = [c for c in required_cols if c not in x.columns]
        if missing_cols:
            raise ValueError(f"{ticker} missing columns: {missing_cols}")

        x = x.sort_values("date").reset_index(drop=True)

        # 确保类型正确
        x["date"] = pd.to_datetime(x["date"], errors="coerce")
        for col in ["open", "high", "low", "close", "volume", "open_interest"]:
            x[col] = pd.to_numeric(x[col], errors="coerce")

        # 如果原表没有 ticker 列，就补上
        if "ticker" not in x.columns:
            x["ticker"] = ticker

        # -------------------------
        # 2) paper里的日收益: r_t = p_t - p_{t-1}
        # -------------------------
        x["ret_1d"] = x["close"].diff()

        # 60-day EWMA volatility on daily additive returns
        x["ewm_vol_60"] = x["ret_1d"].ewm(
            span=60,
            adjust=False,
            min_periods=60
        ).std()

        x["ewm_vol_60"] = x["ewm_vol_60"].replace(0, np.nan)

        # -------------------------
        # 3) normalized close price series
        # 这里用 60-day rolling z-score
        # -------------------------
        close_mean_60 = x["close"].rolling(window=60, min_periods=60).mean()
        close_std_60 = x["close"].rolling(window=60, min_periods=60).std()
        close_std_60 = close_std_60.replace(0, np.nan)

        x["close_norm"] = (x["close"] - close_mean_60) / close_std_60

        # -------------------------
        # 4) 多周期 return features
        # paper: 1m, 2m, 3m, 1y
        # 并用 sigma_t * sqrt(h) 标准化
        # -------------------------
        horizons = {
            "ret_1m": 21,
            "ret_2m": 42,
            "ret_3m": 63,
            "ret_1y": 252
        }

        for feat, h in horizons.items():
            raw_col = f"{feat}_raw"
            x[raw_col] = x["close"] - x["close"].shift(h)
            x[feat] = x[raw_col] / (x["ewm_vol_60"] * np.sqrt(h))

        # -------------------------
        # 5) MACD features
        # paper:
        # q_t = (EMA(S) - EMA(L)) / std(p_{t-63:t})
        # MACD_t = q_t / std(q_{t-252:t})
        # -------------------------
        price_std_63 = x["close"].rolling(window=63, min_periods=63).std()
        price_std_63 = price_std_63.replace(0, np.nan)

        macd_pairs = [(8, 24), (16, 48), (32, 96)]
        macd_cols = []

        for short_span, long_span in macd_pairs:
            ema_s = x["close"].ewm(span=short_span, adjust=False, min_periods=short_span).mean()
            ema_l = x["close"].ewm(span=long_span, adjust=False, min_periods=long_span).mean()

            q_col = f"q_{short_span}_{long_span}"
            macd_col = f"macd_{short_span}_{long_span}"

            x[q_col] = (ema_s - ema_l) / price_std_63

            q_std_252 = x[q_col].rolling(window=252, min_periods=252).std()
            q_std_252 = q_std_252.replace(0, np.nan)

            x[macd_col] = x[q_col] / q_std_252
            macd_cols.append(macd_col)

        x["macd_avg"] = x[macd_cols].mean(axis=1)

        # -------------------------
        # 6) Wilder RSI(30)
        # -------------------------
        delta = x["close"].diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)

        avg_gain = gain.ewm(alpha=1/30, adjust=False, min_periods=30).mean()
        avg_loss = loss.ewm(alpha=1/30, adjust=False, min_periods=30).mean()

        rs = avg_gain / avg_loss.replace(0, np.nan)
        x["rsi_30"] = 100 - (100 / (1 + rs))

        # 特殊情况处理
        x.loc[(avg_loss == 0) & (avg_gain > 0), "rsi_30"] = 100
        x.loc[(avg_loss == 0) & (avg_gain == 0), "rsi_30"] = 50

        # -------------------------
        # 7) 保留常用列
        # -------------------------
        keep_cols = [
            "date", "ticker",
            "open", "high", "low", "close", "volume", "open_interest",
            "ret_1d", "ewm_vol_60", "close_norm",
            "ret_1m_raw", "ret_2m_raw", "ret_3m_raw", "ret_1y_raw",
            "ret_1m", "ret_2m", "ret_3m", "ret_1y",
            "macd_8_24", "macd_16_48", "macd_32_96", "macd_avg",
            "rsi_30"
        ]

        x = x[keep_cols].copy()

        # -------------------------
        # 8) 可选 dropna
        # -------------------------
        core_cols = [
            "ewm_vol_60", "close_norm",
            "ret_1m", "ret_2m", "ret_3m", "ret_1y",
            "macd_8_24", "macd_16_48", "macd_32_96",
            "rsi_30"
        ]

        if dropna:
            x = x.dropna(subset=core_cols).reset_index(drop=True)

        feature_dict[ticker] = x

    feature_panel = pd.concat(feature_dict.values(), ignore_index=True)
    feature_panel = feature_panel.sort_values(["ticker", "date"]).reset_index(drop=True)

    return feature_dict, feature_panel

In [3]:
import numpy as np
import pandas as pd


def make_state_tensor_single(
    df,
    feature_cols,
    window=60,
    return_dates=True,
    return_current_row=False
):
    """
    Convert one ticker's daily feature table into rolling state tensors.

    Each sample at time t uses the previous `window` observations:
        X_t = features[t-window : t]

    So the aligned current row is row t.
    """
    x = df.copy()
    x = x.sort_values("date").reset_index(drop=True)

    # 只保留 feature 完整的行
    x = x.dropna(subset=feature_cols).reset_index(drop=True)

    X = []
    dates = []
    rows = []

    for i in range(window, len(x)):
        state = x.loc[i - window:i - 1, feature_cols].to_numpy(dtype=np.float32)
        X.append(state)
        dates.append(x.loc[i, "date"])
        rows.append(x.loc[i].to_dict())

    if len(X) == 0:
        X = np.empty((0, window, len(feature_cols)), dtype=np.float32)
    else:
        X = np.stack(X).astype(np.float32)

    outputs = [X]

    if return_dates:
        outputs.append(np.array(dates))

    if return_current_row:
        aligned_df = pd.DataFrame(rows).reset_index(drop=True)
        outputs.append(aligned_df)

    if len(outputs) == 1:
        return outputs[0]
    return tuple(outputs)


def make_state_tensors_all(feature_dict, feature_cols, window=60):
    """
    Build rolling state tensors for all tickers.
    """
    state_dict = {}

    for ticker, df in feature_dict.items():
        X, dates, aligned_df = make_state_tensor_single(
            df,
            feature_cols=feature_cols,
            window=window,
            return_dates=True,
            return_current_row=True
        )

        state_dict[ticker] = {
            "X": X,
            "dates": dates,
            "aligned_df": aligned_df
        }

    return state_dict

In [4]:
def build_baselines(df):

    out = df.copy()

    # --- Long Only ---
    out["long_only"] = 1.0

    # --- Sign(R) ---
    ret_1y = out["close"] - out["close"].shift(252)
    out["sign"] = np.sign(ret_1y)

    # --- MACD ---
    macd = out["macd_avg"]
    out["macd_signal"] = macd * np.exp(-macd**2 / 4) / 0.89

    # --- 时间对齐 ---
    out["long_only"] = out["long_only"].shift(1)
    out["sign"] = out["sign"].shift(1)
    out["macd_signal"] = out["macd_signal"].shift(1)

    return out

In [5]:
import numpy as np
import pandas as pd


def compute_pnl(
    df,
    signal_col,
    price_col="close",
    ret_col="ret_1d",
    vol_col="ewm_vol_60",
    sigma_target=0.064, ##注意假设
    bp=0.0020,
    mu=1.0,
    dropna=False
):
    """
    Compute paper-style daily PnL for one strategy on one contract.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain at least:
        [price_col, ret_col, vol_col, signal_col]
    signal_col : str
        Column name of strategy position A_t in [-1, 1]
    price_col : str
        Usually 'close'
    ret_col : str
        Usually 'ret_1d' = p_t - p_{t-1}
    vol_col : str
        Usually 'ewm_vol_60'
    sigma_target : float
        sigma_tgt in the paper
    bp : float
        Cost rate. Paper Table 1 uses 0.0020 in training. :contentReference[oaicite:1]{index=1}
    mu : float
        Per-contract scaling factor. Paper sets mu = 1. :contentReference[oaicite:2]{index=2}
    dropna : bool
        If True, drop rows where key outputs are NaN

    Returns
    -------
    out : pd.DataFrame
        Original df plus:
        signal_prev
        vol_prev
        vol_prev2
        scaled_pos_prev
        scaled_pos_prev2
        gross_pnl
        trading_cost
        net_pnl
    """
    out = df.copy()

    # A_{t-1}, sigma_{t-1}, sigma_{t-2}, p_{t-1}
    out["signal_prev"] = out[signal_col].shift(1)
    out["signal_prev2"] = out[signal_col].shift(2)

    out["vol_prev"] = out[vol_col].shift(1)
    out["vol_prev2"] = out[vol_col].shift(2)

    out["price_prev"] = out[price_col].shift(1)

    # 防止除零
    out["vol_prev"] = out["vol_prev"].replace(0, np.nan)
    out["vol_prev2"] = out["vol_prev2"].replace(0, np.nan)

    # sigma_target / sigma * A
    out["scaled_pos_prev"] = sigma_target / out["vol_prev"] * out["signal_prev"]
    out["scaled_pos_prev2"] = sigma_target / out["vol_prev2"] * out["signal_prev2"]

    # gross pnl = mu * scaled_pos_{t-1} * r_t
    out["gross_pnl"] = mu * out["scaled_pos_prev"] * out[ret_col]

    # trading cost = mu * bp * p_{t-1} * |scaled_pos_{t-1} - scaled_pos_{t-2}|
    out["trading_cost"] = (
        mu
        * bp
        * out["price_prev"]
        * (out["scaled_pos_prev"] - out["scaled_pos_prev2"]).abs()
    )

    # net pnl
    out["net_pnl"] = out["gross_pnl"] - out["trading_cost"]

    if dropna:
        out = out.dropna(
            subset=[
                "scaled_pos_prev",
                "scaled_pos_prev2",
                "gross_pnl",
                "trading_cost",
                "net_pnl",
            ]
        ).reset_index(drop=True)

    return out

In [6]:
def build_portfolio_pnl(pnl_by_ticker, pnl_col="net_pnl"):
    """
    pnl_by_ticker: dict[ticker] = DataFrame returned by compute_pnl
    """
    frames = []
    for ticker, df in pnl_by_ticker.items():
        tmp = df[["date", pnl_col]].copy()
        tmp["ticker"] = ticker
        frames.append(tmp)

    panel = pd.concat(frames, ignore_index=True)

    port = (
        panel.groupby("date", as_index=False)[pnl_col]
        .mean()
        .rename(columns={pnl_col: "portfolio_pnl"})
    )

    return panel, port

In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal


# ============================================================
# Paper-aligned defaults (only what is explicitly stated)
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LR_ACTOR = 1e-4       # paper Table 1
LR_CRITIC = 1e-3      # paper Table 1
GAMMA = 0.3           # paper Table 1
BATCH_SIZE = 128      # paper Table 1
BP = 0.0020           # paper Table 1
MU = 1.0              # paper Eq.4 text
WINDOW = 60           # past 60 observations in state
HIDDEN_SIZES = [64, 32]   # paper Section 4.3
LEAKY_RELU_SLOPE = 0.01


# ============================================================
# 1) Two-layer LSTM backbone
#    Paper says:
#      - LSTM for actor and critic
#      - two-layer LSTM
#      - 64 and 32 units
#      - Leaky-ReLU activation
# ============================================================
class StackedLSTMBackbone(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64, 32)):
        super().__init__()
        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_sizes[0],
            num_layers=1,
            batch_first=True,
        )
        self.lstm2 = nn.LSTM(
            input_size=hidden_sizes[0],
            hidden_size=hidden_sizes[1],
            num_layers=1,
            batch_first=True,
        )
        self.act = nn.LeakyReLU(LEAKY_RELU_SLOPE)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out1, _ = self.lstm1(x)
        out1 = self.act(out1)

        out2, _ = self.lstm2(out1)
        out2 = self.act(out2)

        # IMPLEMENTATION ASSUMPTION:
        # paper does not specify pooling/readout method from LSTM.
        # Standard choice: use last time-step hidden representation.
        return out2[:, -1, :]   # (batch, 32)


# ============================================================
# 2) Actor: continuous action in [-1, 1]
#    Paper states A2C uses continuous action space [−1,1],
#    but does not specify the exact policy distribution.
# ============================================================
class ActorContinuous(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64, 32)):
        super().__init__()
        self.backbone = StackedLSTMBackbone(input_size, hidden_sizes)
        self.mu_head = nn.Linear(hidden_sizes[-1], 1)

        # IMPLEMENTATION ASSUMPTION:
        # paper does not specify how to parameterize continuous action policy.
        # We use a Gaussian policy with a trainable global log_std.
        self.log_std = nn.Parameter(torch.tensor(-0.5, dtype=torch.float32))

    def forward(self, x):
        h = self.backbone(x)
        mu = torch.tanh(self.mu_head(h))   # action mean constrained to [-1,1]
        std = torch.exp(self.log_std).clamp(min=1e-4, max=2.0)
        return mu, std

    def sample_action(self, x):
        mu, std = self.forward(x)
        dist = Normal(mu, std)

        z = dist.rsample()
        action = torch.tanh(z)

        # tanh squash correction
        log_prob = dist.log_prob(z) - torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1)

        entropy = dist.entropy().sum(dim=-1)
        return action, log_prob, entropy

    def deterministic_action(self, x):
        mu, _ = self.forward(x)
        return torch.tanh(mu)


# ============================================================
# 3) Critic: state value V(s)
# ============================================================
class CriticValue(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64, 32)):
        super().__init__()
        self.backbone = StackedLSTMBackbone(input_size, hidden_sizes)
        self.value_head = nn.Linear(hidden_sizes[-1], 1)

    def forward(self, x):
        h = self.backbone(x)
        v = self.value_head(h)
        return v.squeeze(-1)


# ============================================================
# 4) Paper-style trading environment
#
#    Expected inputs:
#      X: shape (n_samples, 60, n_features)
#      aligned_df: DataFrame aligned with X, containing at least:
#        date, close, ret_1d, ewm_vol_60
#
#    Reward logic follows the paper's Eq.(4), with timing written as:
#      choose A_t from state at time t
#      receive reward on t -> t+1
#
#    sigma_target is left as an input argument because the paper
#    includes sigma_tgt in Eq.(4) but does not fix one universal value.
# ============================================================
class PaperTradingEnv:
    def __init__(
        self,
        X,
        aligned_df,
        sigma_target,
        bp=BP,
        mu=MU,
    ):
        self.X = np.asarray(X, dtype=np.float32)
        self.df = aligned_df.reset_index(drop=True).copy()
        self.sigma_target = sigma_target
        self.bp = bp
        self.mu = mu

        required_cols = ["date", "close", "ret_1d", "ewm_vol_60"]
        missing = [c for c in required_cols if c not in self.df.columns]
        if missing:
            raise ValueError(f"aligned_df missing required columns: {missing}")

        if len(self.X) != len(self.df):
            raise ValueError("X and aligned_df must have the same number of samples.")

        self.n = len(self.df)
        self.reset()

    def reset(self):
        self.t = 0
        self.prev_action = 0.0
        return self.X[self.t]

    def step(self, action):
        action = float(np.clip(action, -1.0, 1.0))

        if self.t >= self.n - 1:
            return self.X[self.t], 0.0, True, {}

        row_t = self.df.iloc[self.t]
        row_tp1 = self.df.iloc[self.t + 1]

        sigma_t = row_t["ewm_vol_60"]
        sigma_tm1 = self.df.iloc[self.t - 1]["ewm_vol_60"] if self.t - 1 >= 0 else np.nan
        price_t = row_t["close"]
        r_tp1 = row_tp1["ret_1d"]

        if pd.isna(sigma_t) or sigma_t == 0 or pd.isna(price_t) or pd.isna(r_tp1):
            reward = np.nan
        else:
            scaled_pos_t = self.sigma_target / sigma_t * action

            if self.t == 0 or pd.isna(sigma_tm1) or sigma_tm1 == 0:
                scaled_pos_tm1 = 0.0
            else:
                scaled_pos_tm1 = self.sigma_target / sigma_tm1 * self.prev_action

            gross = self.mu * scaled_pos_t * r_tp1
            cost = self.mu * self.bp * price_t * abs(scaled_pos_t - scaled_pos_tm1)
            reward = gross - cost

        self.prev_action = action
        self.t += 1
        done = self.t >= self.n - 1

        next_state = self.X[self.t]
        info = {
            "date": row_tp1["date"],
            "reward": reward,
        }
        return next_state, float(0.0 if pd.isna(reward) else reward), done, info


# ============================================================
# 5) Collect synchronous rollouts across environments
# ============================================================
def collect_rollout(envs, actor, critic, rollout_steps, device=DEVICE):
    states = []
    actions = []
    rewards = []
    dones = []
    values = []

    if not hasattr(collect_rollout, "_current_states") or len(collect_rollout._current_states) != len(envs):
        collect_rollout._current_states = [env.reset() for env in envs]

    current_states = collect_rollout._current_states

    for _ in range(rollout_steps):
        state_batch = torch.tensor(np.stack(current_states), dtype=torch.float32, device=device)

        with torch.no_grad():
            action_batch, _, _ = actor.sample_action(state_batch)
            value_batch = critic(state_batch)

        next_states = []
        reward_batch = []
        done_batch = []

        for i, env in enumerate(envs):
            a = action_batch[i].item()
            ns, r, d, _ = env.step(a)

            if d:
                ns = env.reset()

            next_states.append(ns)
            reward_batch.append(r)
            done_batch.append(float(d))

        states.append(state_batch)
        actions.append(action_batch.squeeze(-1))
        rewards.append(torch.tensor(reward_batch, dtype=torch.float32, device=device))
        dones.append(torch.tensor(done_batch, dtype=torch.float32, device=device))
        values.append(value_batch)

        current_states = next_states

    collect_rollout._current_states = current_states

    last_state_batch = torch.tensor(np.stack(current_states), dtype=torch.float32, device=device)
    with torch.no_grad():
        last_values = critic(last_state_batch)

    batch = {
        "states": torch.stack(states),       # (T, N, 60, F)
        "actions": torch.stack(actions),     # (T, N)
        "rewards": torch.stack(rewards),     # (T, N)
        "dones": torch.stack(dones),         # (T, N)
        "values": torch.stack(values),       # (T, N)
        "last_values": last_values,          # (N,)
    }
    return batch


# ============================================================
# 6) Returns / advantages
# ============================================================
def compute_returns_and_advantages(rewards, dones, values, last_values, gamma=GAMMA):
    T, N = rewards.shape
    returns = torch.zeros_like(rewards)
    running = last_values

    for t in reversed(range(T)):
        running = rewards[t] + gamma * (1.0 - dones[t]) * running
        returns[t] = running

    advantages = returns - values
    return returns, advantages


# ============================================================
# 7) Main A2C trainer
#
#    Checkpoint / resume is added as an engineering feature.
#    It does not change the paper's model or hyperparameters.
# ============================================================
class PaperA2CTrainer:
    def __init__(
        self,
        n_features,
        lr_actor=LR_ACTOR,
        lr_critic=LR_CRITIC,
        gamma=GAMMA,
        entropy_coef=1e-3,    # IMPLEMENTATION ASSUMPTION: not specified in paper
        max_grad_norm=1.0,    # IMPLEMENTATION ASSUMPTION: not specified in paper
        device=DEVICE,
    ):
        self.device = device
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

        self.actor = ActorContinuous(n_features, HIDDEN_SIZES).to(device)
        self.critic = CriticValue(n_features, HIDDEN_SIZES).to(device)

        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optim = torch.optim.Adam(self.critic.parameters(), lr=lr_critic)

        self.train_log = []

    def update(self, batch):
        T, N, W, feat_dim = batch["states"].shape
        states_flat = batch["states"].reshape(T * N, W, feat_dim)

        returns, advantages = compute_returns_and_advantages(
            batch["rewards"],
            batch["dones"],
            batch["values"],
            batch["last_values"],
            gamma=self.gamma,
        )

        returns_flat = returns.reshape(T * N)
        advantages_flat = advantages.reshape(T * N)
        advantages_flat = (advantages_flat - advantages_flat.mean()) / (advantages_flat.std() + 1e-8)

        # ---- Critic ----
        values_pred = self.critic(states_flat)
        critic_loss = F.mse_loss(values_pred, returns_flat)

        self.critic_optim.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), self.max_grad_norm)
        self.critic_optim.step()

        # ---- Actor ----
        mu, std = self.actor(states_flat)
        dist = Normal(mu, std)

        actions_flat = batch["actions"].reshape(T * N).unsqueeze(-1).clamp(-0.999, 0.999)
        z = 0.5 * torch.log((1 + actions_flat) / (1 - actions_flat))
        log_probs = dist.log_prob(z) - torch.log(1 - actions_flat.pow(2) + 1e-6)
        log_probs = log_probs.sum(dim=-1)

        entropy = dist.entropy().sum(dim=-1).mean()
        actor_loss = -(log_probs * advantages_flat.detach()).mean() - self.entropy_coef * entropy

        self.actor_optim.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), self.max_grad_norm)
        self.actor_optim.step()

        out = {
            "actor_loss": float(actor_loss.item()),
            "critic_loss": float(critic_loss.item()),
            "entropy": float(entropy.item()),
            "avg_return": float(returns_flat.mean().item()),
            "avg_reward": float(batch["rewards"].mean().item()),
        }
        self.train_log.append(out)
        return out

    def save_checkpoint(self, path, extra_state=None):
        ckpt = {
            "actor_state_dict": self.actor.state_dict(),
            "critic_state_dict": self.critic.state_dict(),
            "actor_optim_state_dict": self.actor_optim.state_dict(),
            "critic_optim_state_dict": self.critic_optim.state_dict(),
            "train_log": self.train_log,
            "gamma": self.gamma,
            "entropy_coef": self.entropy_coef,
            "max_grad_norm": self.max_grad_norm,
        }
        if extra_state is not None:
            ckpt["extra_state"] = extra_state

        torch.save(ckpt, path)

    def load_checkpoint(self, path, map_location=None):
        ckpt = torch.load(path, map_location=map_location or self.device)

        self.actor.load_state_dict(ckpt["actor_state_dict"])
        self.critic.load_state_dict(ckpt["critic_state_dict"])
        self.actor_optim.load_state_dict(ckpt["actor_optim_state_dict"])
        self.critic_optim.load_state_dict(ckpt["critic_optim_state_dict"])
        self.train_log = ckpt.get("train_log", [])

        return ckpt

    def fit(
        self,
        envs,
        n_updates=1000,
        rollout_steps=32,
        log_every=50,
        checkpoint_path=None,
        checkpoint_every=50,
        start_update=0,
    ):
        # avoid rollout state leaking across asset classes / separate runs
        if hasattr(collect_rollout, "_current_states"):
            del collect_rollout._current_states

        for update_idx in range(start_update + 1, start_update + n_updates + 1):
            batch = collect_rollout(
                envs=envs,
                actor=self.actor,
                critic=self.critic,
                rollout_steps=rollout_steps,
                device=self.device,
            )

            metrics = self.update(batch)

            if update_idx % log_every == 0:
                print(
                    f"[{update_idx:4d}] "
                    f"actor_loss={metrics['actor_loss']:.4f} "
                    f"critic_loss={metrics['critic_loss']:.4f} "
                    f"avg_reward={metrics['avg_reward']:.6f} "
                    f"entropy={metrics['entropy']:.4f}"
                )

            if checkpoint_path is not None and update_idx % checkpoint_every == 0:
                self.save_checkpoint(
                    checkpoint_path,
                    extra_state={
                        "update_idx": update_idx,
                        "rollout_steps": rollout_steps,
                    }
                )
                print(f"checkpoint saved to {checkpoint_path}")

        return self


# ============================================================
# 8) Build envs from state_dict
# ============================================================
def build_envs_from_state_dict(
    state_dict,
    tickers=None,
    sigma_target=None,
    bp=BP,
    mu=MU,
):
    if sigma_target is None:
        raise ValueError("sigma_target must be provided explicitly because the paper does not fix a universal value.")

    envs = []
    selected = tickers if tickers is not None else list(state_dict.keys())

    for ticker in selected:
        if ticker not in state_dict:
            continue

        X = state_dict[ticker]["X"]
        aligned_df = state_dict[ticker]["aligned_df"].copy()

        if not isinstance(aligned_df, pd.DataFrame):
            continue
        if len(aligned_df) == 0 or len(X) == 0:
            continue
        if not all(c in aligned_df.columns for c in ["close", "ret_1d", "ewm_vol_60"]):
            continue

        usable = aligned_df[["close", "ret_1d", "ewm_vol_60"]].notna().all(axis=1).values
        X = X[usable]
        aligned_df = aligned_df.loc[usable].reset_index(drop=True)

        if len(X) > 2:
            envs.append(
                PaperTradingEnv(
                    X=X,
                    aligned_df=aligned_df,
                    sigma_target=sigma_target,
                    bp=bp,
                    mu=mu,
                )
            )
    return envs

In [8]:
import numpy as np
import pandas as pd


# ============================================================
# 1) Paper reference values: Table 2 and Table 3
#    Source: Deep Reinforcement Learning for Trading, p.8 and Appendix B p.16.
# ============================================================

PAPER_METRICS = ["E(R)", "Std(R)", "DD", "Sharpe", "Sortino", "MDD", "Calmar", "% +Ret", "Ave.P/Ave.L"]

PAPER_TABLE2 = {
    "Commodity": {
        "Long":    {"E(R)": -0.710, "Std(R)": 0.979, "DD": 0.604, "Sharpe": -0.726, "Sortino": -1.177, "MDD": 0.350, "Calmar": -0.140, "% +Ret": 0.473, "Ave.P/Ave.L": 0.989},
        "Sign(R)": {"E(R)":  0.347, "Std(R)": 0.980, "DD": 0.572, "Sharpe":  0.354, "Sortino":  0.606, "MDD": 0.116, "Calmar":  0.119, "% +Ret": 0.494, "Ave.P/Ave.L": 1.084},
        "MACD":    {"E(R)": -0.171, "Std(R)": 0.978, "DD": 0.584, "Sharpe": -0.175, "Sortino": -0.293, "MDD": 0.190, "Calmar": -0.060, "% +Ret": 0.486, "Ave.P/Ave.L": 1.026},
        "DQN":     {"E(R)":  0.703, "Std(R)": 0.973, "DD": 0.552, "Sharpe":  0.723, "Sortino":  1.275, "MDD": 0.066, "Calmar":  0.501, "% +Ret": 0.498, "Ave.P/Ave.L": 1.135},
        "PG":      {"E(R)":  0.062, "Std(R)": 0.982, "DD": 0.585, "Sharpe":  0.063, "Sortino":  0.106, "MDD": 0.039, "Calmar":  0.023, "% +Ret": 0.495, "Ave.P/Ave.L": 1.029},
        "A2C":     {"E(R)":  0.223, "Std(R)": 0.955, "DD": 0.559, "Sharpe":  0.234, "Sortino":  0.399, "MDD": 0.141, "Calmar":  0.091, "% +Ret": 0.487, "Ave.P/Ave.L": 1.093},
    },
    "Equity Index": {
        "Long":    {"E(R)": 0.668, "Std(R)": 0.970, "DD": 0.606, "Sharpe": 0.688, "Sortino": 1.102, "MDD": 0.132, "Calmar": 0.509, "% +Ret": 0.542, "Ave.P/Ave.L": 0.948},
        "Sign(R)": {"E(R)": 0.228, "Std(R)": 0.966, "DD": 0.610, "Sharpe": 0.236, "Sortino": 0.374, "MDD": 0.344, "Calmar": 0.077, "% +Ret": 0.528, "Ave.P/Ave.L": 0.930},
        "MACD":    {"E(R)": 0.016, "Std(R)": 0.962, "DD": 0.618, "Sharpe": 0.017, "Sortino": 0.027, "MDD": 0.311, "Calmar": 0.006, "% +Ret": 0.519, "Ave.P/Ave.L": 0.927},
        "DQN":     {"E(R)": 0.629, "Std(R)": 0.970, "DD": 0.606, "Sharpe": 0.648, "Sortino": 1.038, "MDD": 0.161, "Calmar": 0.381, "% +Ret": 0.541, "Ave.P/Ave.L": 0.944},
        "PG":      {"E(R)": 0.432, "Std(R)": 0.967, "DD": 0.605, "Sharpe": 0.447, "Sortino": 0.714, "MDD": 0.242, "Calmar": 0.185, "% +Ret": 0.529, "Ave.P/Ave.L": 0.960},
        "A2C":     {"E(R)": 0.473, "Std(R)": 0.929, "DD": 0.593, "Sharpe": 0.510, "Sortino": 0.798, "MDD": 0.124, "Calmar": 0.328, "% +Ret": 0.533, "Ave.P/Ave.L": 0.962},
    },
    "Fixed Income": {
        "Long":    {"E(R)": 0.680, "Std(R)": 0.975, "DD": 0.576, "Sharpe": 0.698, "Sortino": 1.180, "MDD": 0.061, "Calmar": 0.444, "% +Ret": 0.515, "Ave.P/Ave.L": 1.054},
        "Sign(R)": {"E(R)": 0.214, "Std(R)": 0.972, "DD": 0.592, "Sharpe": 0.221, "Sortino": 0.363, "MDD": 0.080, "Calmar": 0.083, "% +Ret": 0.504, "Ave.P/Ave.L": 1.019},
        "MACD":    {"E(R)": 0.219, "Std(R)": 0.967, "DD": 0.579, "Sharpe": 0.228, "Sortino": 0.380, "MDD": 0.065, "Calmar": 0.123, "% +Ret": 0.486, "Ave.P/Ave.L": 1.101},
        "DQN":     {"E(R)": 0.908, "Std(R)": 0.972, "DD": 0.562, "Sharpe": 0.935, "Sortino": 1.617, "MDD": 0.062, "Calmar": 0.543, "% +Ret": 0.515, "Ave.P/Ave.L": 1.098},
        "PG":      {"E(R)": 0.705, "Std(R)": 0.974, "DD": 0.576, "Sharpe": 0.724, "Sortino": 1.225, "MDD": 0.061, "Calmar": 0.436, "% +Ret": 0.517, "Ave.P/Ave.L": 1.052},
        "A2C":     {"E(R)": 0.699, "Std(R)": 0.979, "DD": 0.582, "Sharpe": 0.714, "Sortino": 1.203, "MDD": 0.067, "Calmar": 0.408, "% +Ret": 0.517, "Ave.P/Ave.L": 1.048},
    },
    "FX": {
        "Long":    {"E(R)": -0.344, "Std(R)": 0.973, "DD": 0.583, "Sharpe": -0.353, "Sortino": -0.590, "MDD": 0.423, "Calmar": -0.097, "% +Ret": 0.491, "Ave.P/Ave.L": 0.979},
        "Sign(R)": {"E(R)": -0.297, "Std(R)": 0.973, "DD": 0.592, "Sharpe": -0.306, "Sortino": -0.502, "MDD": 0.434, "Calmar": -0.111, "% +Ret": 0.499, "Ave.P/Ave.L": 0.954},
        "MACD":    {"E(R)":  0.006, "Std(R)": 0.970, "DD": 0.582, "Sharpe":  0.007, "Sortino":  0.011, "MDD": 0.329, "Calmar":  0.002, "% +Ret": 0.493, "Ave.P/Ave.L": 1.029},
        "DQN":     {"E(R)":  0.528, "Std(R)": 0.967, "DD": 0.553, "Sharpe":  0.546, "Sortino":  0.955, "MDD": 0.183, "Calmar":  0.313, "% +Ret": 0.510, "Ave.P/Ave.L": 1.051},
        "PG":      {"E(R)":  0.248, "Std(R)": 0.967, "DD": 0.566, "Sharpe":  0.257, "Sortino":  0.438, "MDD": 0.240, "Calmar":  0.124, "% +Ret": 0.506, "Ave.P/Ave.L": 1.021},
        "A2C":     {"E(R)":  0.316, "Std(R)": 0.963, "DD": 0.563, "Sharpe":  0.328, "Sortino":  0.561, "MDD": 0.165, "Calmar":  0.201, "% +Ret": 0.507, "Ave.P/Ave.L": 1.026},
    },
    "All": {
        "Long":    {"E(R)": 0.055, "Std(R)": 0.975, "DD": 0.598, "Sharpe": 0.058, "Sortino": 0.092, "MDD": 0.071, "Calmar": 0.013, "% +Ret": 0.520, "Ave.P/Ave.L": 0.933},
        "Sign(R)": {"E(R)": 0.429, "Std(R)": 0.972, "DD": 0.582, "Sharpe": 0.441, "Sortino": 0.737, "MDD": 0.038, "Calmar": 0.201, "% +Ret": 0.510, "Ave.P/Ave.L": 1.031},
        "MACD":    {"E(R)": 0.089, "Std(R)": 0.978, "DD": 0.582, "Sharpe": 0.091, "Sortino": 0.153, "MDD": 0.008, "Calmar": 0.035, "% +Ret": 0.493, "Ave.P/Ave.L": 1.043},
        "DQN":     {"E(R)": 1.258, "Std(R)": 0.976, "DD": 0.567, "Sharpe": 1.288, "Sortino": 2.220, "MDD": 0.002, "Calmar": 1.025, "% +Ret": 0.543, "Ave.P/Ave.L": 1.043},
        "PG":      {"E(R)": 0.740, "Std(R)": 0.980, "DD": 0.593, "Sharpe": 0.754, "Sortino": 1.247, "MDD": 0.012, "Calmar": 0.480, "% +Ret": 0.533, "Ave.P/Ave.L": 0.990},
        "A2C":     {"E(R)": 1.024, "Std(R)": 0.975, "DD": 0.573, "Sharpe": 1.050, "Sortino": 1.785, "MDD": 0.007, "Calmar": 0.685, "% +Ret": 0.538, "Ave.P/Ave.L": 1.021},
    }
}

PAPER_TABLE3 = {
    "Commodity": {
        "Long":    {"E(R)": -0.298, "Std(R)": 0.412, "DD": 0.258, "Sharpe": -0.723, "Sortino": -1.152, "MDD": 0.248, "Calmar": -0.130, "% +Ret": 0.473, "Ave.P/Ave.L": 0.987},
        "Sign(R)": {"E(R)":  0.101, "Std(R)": 0.312, "DD": 0.185, "Sharpe":  0.325, "Sortino":  0.548, "MDD": 0.082, "Calmar":  0.115, "% +Ret": 0.494, "Ave.P/Ave.L": 1.081},
        "MACD":    {"E(R)": -0.039, "Std(R)": 0.227, "DD": 0.136, "Sharpe": -0.174, "Sortino": -0.290, "MDD": 0.132, "Calmar": -0.059, "% +Ret": 0.486, "Ave.P/Ave.L": 1.024},
        "DQN":     {"E(R)":  0.187, "Std(R)": 0.301, "DD": 0.173, "Sharpe":  0.623, "Sortino":  1.085, "MDD": 0.039, "Calmar":  0.413, "% +Ret": 0.498, "Ave.P/Ave.L": 1.119},
        "PG":      {"E(R)":  0.013, "Std(R)": 0.287, "DD": 0.172, "Sharpe":  0.047, "Sortino":  0.078, "MDD": 0.072, "Calmar":  0.017, "% +Ret": 0.495, "Ave.P/Ave.L": 1.026},
        "A2C":     {"E(R)":  0.072, "Std(R)": 0.163, "DD": 0.098, "Sharpe":  0.440, "Sortino":  0.729, "MDD": 0.099, "Calmar":  0.161, "% +Ret": 0.487, "Ave.P/Ave.L": 1.151},
    },
    "Equity Index": {
        "Long":    {"E(R)": 0.504, "Std(R)": 0.928, "DD": 0.606, "Sharpe": 0.543, "Sortino": 0.831, "MDD": 0.127, "Calmar": 0.466, "% +Ret": 0.541, "Ave.P/Ave.L": 0.928},
        "Sign(R)": {"E(R)": 0.168, "Std(R)": 0.799, "DD": 0.526, "Sharpe": 0.211, "Sortino": 0.319, "MDD": 0.299, "Calmar": 0.075, "% +Ret": 0.528, "Ave.P/Ave.L": 0.928},
        "MACD":    {"E(R)":-0.068, "Std(R)": 0.586, "DD": 0.385, "Sharpe":-0.117, "Sortino":-0.178, "MDD": 0.351, "Calmar":-0.041, "% +Ret": 0.519, "Ave.P/Ave.L": 0.904},
        "DQN":     {"E(R)": 0.461, "Std(R)": 0.933, "DD": 0.611, "Sharpe": 0.494, "Sortino": 0.754, "MDD": 0.170, "Calmar": 0.308, "% +Ret": 0.541, "Ave.P/Ave.L": 0.922},
        "PG":      {"E(R)": 0.320, "Std(R)": 0.875, "DD": 0.574, "Sharpe": 0.366, "Sortino": 0.558, "MDD": 0.211, "Calmar": 0.183, "% +Ret": 0.529, "Ave.P/Ave.L": 0.949},
        "A2C":     {"E(R)": 0.293, "Std(R)": 0.629, "DD": 0.427, "Sharpe": 0.466, "Sortino": 0.686, "MDD": 0.193, "Calmar": 0.214, "% +Ret": 0.533, "Ave.P/Ave.L": 0.965},
    },
    "Fixed Income": {
        "Long":    {"E(R)": 0.605, "Std(R)": 0.939, "DD": 0.561, "Sharpe": 0.645, "Sortino": 1.081, "MDD": 0.108, "Calmar": 0.455, "% +Ret": 0.515, "Ave.P/Ave.L": 1.048},
        "Sign(R)": {"E(R)": 0.189, "Std(R)": 0.795, "DD": 0.496, "Sharpe": 0.237, "Sortino": 0.381, "MDD": 0.165, "Calmar": 0.103, "% +Ret": 0.504, "Ave.P/Ave.L": 1.024},
        "MACD":    {"E(R)": 0.136, "Std(R)": 0.609, "DD": 0.367, "Sharpe": 0.224, "Sortino": 0.371, "MDD": 0.124, "Calmar": 0.131, "% +Ret": 0.485, "Ave.P/Ave.L": 1.102},
        "DQN":     {"E(R)": 0.734, "Std(R)": 0.862, "DD": 0.508, "Sharpe": 0.851, "Sortino": 1.445, "MDD": 0.118, "Calmar": 0.469, "% +Ret": 0.515, "Ave.P/Ave.L": 1.086},
        "PG":      {"E(R)": 0.624, "Std(R)": 0.938, "DD": 0.561, "Sharpe": 0.665, "Sortino": 1.113, "MDD": 0.109, "Calmar": 0.443, "% +Ret": 0.517, "Ave.P/Ave.L": 1.043},
        "A2C":     {"E(R)": 0.852, "Std(R)": 1.345, "DD": 0.806, "Sharpe": 0.633, "Sortino": 1.057, "MDD": 0.128, "Calmar": 0.397, "% +Ret": 0.517, "Ave.P/Ave.L": 1.039},
    },
    "FX": {
        "Long":    {"E(R)":-0.198, "Std(R)": 0.472, "DD": 0.285, "Sharpe":-0.420, "Sortino":-0.696, "MDD": 0.219, "Calmar":-0.101, "% +Ret": 0.491, "Ave.P/Ave.L": 0.966},
        "Sign(R)": {"E(R)":-0.113, "Std(R)": 0.551, "DD": 0.341, "Sharpe":-0.207, "Sortino":-0.332, "MDD": 0.170, "Calmar":-0.071, "% +Ret": 0.499, "Ave.P/Ave.L": 0.968},
        "MACD":    {"E(R)": 0.016, "Std(R)": 0.424, "DD": 0.259, "Sharpe": 0.037, "Sortino": 0.061, "MDD": 0.156, "Calmar": 0.016, "% +Ret": 0.493, "Ave.P/Ave.L": 1.034},
        "DQN":     {"E(R)": 0.272, "Std(R)": 0.487, "DD": 0.280, "Sharpe": 0.560, "Sortino": 0.972, "MDD": 0.085, "Calmar": 0.326, "% +Ret": 0.510, "Ave.P/Ave.L": 1.058},
        "PG":      {"E(R)": 0.157, "Std(R)": 0.533, "DD": 0.312, "Sharpe": 0.295, "Sortino": 0.503, "MDD": 0.098, "Calmar": 0.148, "% +Ret": 0.506, "Ave.P/Ave.L": 1.029},
        "A2C":     {"E(R)": 0.159, "Std(R)": 0.455, "DD": 0.267, "Sharpe": 0.349, "Sortino": 0.592, "MDD": 0.081, "Calmar": 0.193, "% +Ret": 0.507, "Ave.P/Ave.L": 1.034},
    },
    "All": {
        "Long":    {"E(R)":-0.013, "Std(R)": 0.363, "DD": 0.230, "Sharpe":-0.036, "Sortino":-0.057, "MDD": 0.037, "Calmar":-0.009, "% +Ret": 0.519, "Ave.P/Ave.L": 0.919},
        "Sign(R)": {"E(R)": 0.086, "Std(R)": 0.296, "DD": 0.186, "Sharpe": 0.291, "Sortino": 0.461, "MDD": 0.016, "Calmar": 0.142, "% +Ret": 0.510, "Ave.P/Ave.L": 1.008},
        "MACD":    {"E(R)":-0.018, "Std(R)": 0.230, "DD": 0.143, "Sharpe":-0.080, "Sortino":-0.129, "MDD": 0.026, "Calmar":-0.029, "% +Ret": 0.493, "Ave.P/Ave.L": 1.013},
        "DQN":     {"E(R)": 0.318, "Std(R)": 0.252, "DD": 0.150, "Sharpe": 1.258, "Sortino": 2.111, "MDD": 0.008, "Calmar": 1.023, "% +Ret": 0.543, "Ave.P/Ave.L": 1.041},
        "PG":      {"E(R)": 0.168, "Std(R)": 0.279, "DD": 0.174, "Sharpe": 0.602, "Sortino": 0.968, "MDD": 0.011, "Calmar": 0.373, "% +Ret": 0.533, "Ave.P/Ave.L": 0.968},
        "A2C":     {"E(R)": 0.214, "Std(R)": 0.221, "DD": 0.134, "Sharpe": 0.969, "Sortino": 1.601, "MDD": 0.009, "Calmar": 0.672, "% +Ret": 0.538, "Ave.P/Ave.L": 1.014},
    }
}


# ============================================================
# 2) Asset-class ticker map from Appendix A
# ============================================================

PAPER_TICKERS_BY_ASSET_CLASS = {
    "Commodity": ["CC","DA","GI","JO","KC","KW","LB","NR","SB","ZA","ZC","ZF","ZG","ZH","ZI","ZK","ZL","ZN","ZO","ZP","ZR","ZT","ZU","ZW","ZZ"],
    "Equity Index": ["CA","EN","ER","ES","LX","MD","SC","SP","XU","XX","YM"],
    "Fixed Income": ["DT","FB","TY","UB","US"],
    "FX": ["AN","BN","CN","DX","FN","JN","MP","NK","SN"],
}
PAPER_TICKERS_BY_ASSET_CLASS["All"] = sorted(sum(PAPER_TICKERS_BY_ASSET_CLASS.values(), []))


# ============================================================
# 3) Metrics exactly matching the paper's list
# ============================================================

def compute_paper_metrics(daily_returns, periods_per_year=252):
    """
    daily_returns: 1D pandas Series of portfolio daily trade returns
    Returns a dict with the 9 metrics listed in the paper.
    """
    r = pd.Series(daily_returns).dropna().astype(float)
    if len(r) == 0:
        return {k: np.nan for k in PAPER_METRICS}

    ann_mean = r.mean() * periods_per_year
    ann_std = r.std(ddof=1) * np.sqrt(periods_per_year)

    neg_r = r[r < 0]
    if len(neg_r) > 1:
        downside_dev = neg_r.std(ddof=1) * np.sqrt(periods_per_year)
    else:
        downside_dev = np.nan

    sharpe = ann_mean / ann_std if pd.notna(ann_std) and ann_std != 0 else np.nan
    sortino = ann_mean / downside_dev if pd.notna(downside_dev) and downside_dev != 0 else np.nan

    cum = r.cumsum()
    running_max = cum.cummax()
    drawdown = running_max - cum
    mdd = drawdown.max() if len(drawdown) else np.nan

    calmar = ann_mean / mdd if pd.notna(mdd) and mdd != 0 else np.nan

    pct_pos = (r > 0).mean()

    pos_mean = r[r > 0].mean() if (r > 0).any() else np.nan
    neg_mean_abs = (-r[r < 0]).mean() if (r < 0).any() else np.nan
    ave_p_ave_l = pos_mean / neg_mean_abs if pd.notna(pos_mean) and pd.notna(neg_mean_abs) and neg_mean_abs != 0 else np.nan

    return {
        "E(R)": ann_mean,
        "Std(R)": ann_std,
        "DD": downside_dev,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "MDD": mdd,
        "Calmar": calmar,
        "% +Ret": pct_pos,
        "Ave.P/Ave.L": ave_p_ave_l,
    }


# ============================================================
# 4) Portfolio-level volatility scaling for Table 2
#    The paper says Table 2 adds an additional portfolio-level
#    volatility scaling layer so different methods have the same target vol.
#    It does not spell out one exact implementation formula here,
#    so this is an implementation assumption.
# ============================================================

def apply_portfolio_volatility_targeting(
    portfolio_df,
    return_col="portfolio_pnl",
    target_vol=0.975,      # implementation choice, chosen to match Table 2 scale
    ewm_span=60,
    periods_per_year=252,
    out_col="portfolio_pnl_vol_scaled"
):
    """
    Adds a portfolio-level volatility scaling layer.

    Assumption:
    scale_t = target_vol / ann_vol_est_t
    where ann_vol_est_t comes from 60-day EWMA std of the portfolio daily returns.
    Then scaled_return_t = scale_{t-1} * return_t
    """
    out = portfolio_df.copy().sort_values("date").reset_index(drop=True)
    r = out[return_col].astype(float)

    daily_vol_est = r.ewm(span=ewm_span, adjust=False, min_periods=ewm_span).std()
    ann_vol_est = daily_vol_est * np.sqrt(periods_per_year)

    scale = target_vol / ann_vol_est.replace(0, np.nan)
    scale = scale.shift(1)  # no look-ahead

    out[out_col] = scale * r
    return out


# ============================================================
# 5) Existing portfolio aggregation, kept generic
# ============================================================

def build_portfolio_pnl(pnl_by_ticker, pnl_col="net_pnl"):
    frames = []
    for ticker, df in pnl_by_ticker.items():
        tmp = df[["date", pnl_col]].copy()
        tmp["ticker"] = ticker
        frames.append(tmp)

    panel = pd.concat(frames, ignore_index=True)

    port = (
        panel.groupby("date", as_index=False)[pnl_col]
        .mean()
        .rename(columns={pnl_col: "portfolio_pnl"})
        .sort_values("date")
        .reset_index(drop=True)
    )

    return panel, port


# ============================================================
# 6) Compute one model's results in Table 2 / Table 3 style
# ============================================================

def compute_model_table2_table3_results(
    pnl_by_ticker,
    periods_per_year=252,
    table2_target_vol=0.975,
    table2_ewm_span=60,
):
    """
    pnl_by_ticker: dict[ticker] -> DataFrame with at least ['date', 'net_pnl']

    Returns:
        {
            "table3_portfolio": raw portfolio df,
            "table2_portfolio": portfolio df after vol targeting,
            "table3_metrics": {...},
            "table2_metrics": {...},
        }
    """
    _, port_raw = build_portfolio_pnl(pnl_by_ticker, pnl_col="net_pnl")

    table3_metrics = compute_paper_metrics(
        port_raw["portfolio_pnl"],
        periods_per_year=periods_per_year
    )

    port_t2 = apply_portfolio_volatility_targeting(
        port_raw,
        return_col="portfolio_pnl",
        target_vol=table2_target_vol,
        ewm_span=table2_ewm_span,
        periods_per_year=periods_per_year,
        out_col="portfolio_pnl_vol_scaled"
    )

    table2_metrics = compute_paper_metrics(
        port_t2["portfolio_pnl_vol_scaled"],
        periods_per_year=periods_per_year
    )

    return {
        "table3_portfolio": port_raw,
        "table2_portfolio": port_t2,
        "table3_metrics": table3_metrics,
        "table2_metrics": table2_metrics,
    }


# ============================================================
# 7) Compare your result vs paper
# ============================================================

def _safe_pct_diff(yours, paper):
    if pd.isna(yours) or pd.isna(paper):
        return np.nan
    if paper == 0:
        return np.nan
    return (yours - paper) / abs(paper) * 100.0


def compare_model_to_paper(
    your_metrics,
    paper_table,
    asset_class,
    model_name
):
    """
    your_metrics: dict metric -> value
    paper_table: PAPER_TABLE2 or PAPER_TABLE3
    """
    paper_metrics = paper_table[asset_class][model_name]

    rows = []
    for m in PAPER_METRICS:
        y = your_metrics.get(m, np.nan)
        p = paper_metrics.get(m, np.nan)
        rows.append({
            "asset_class": asset_class,
            "model": model_name,
            "metric": m,
            "your_value": y,
            "paper_value": p,
            "pct_diff_vs_paper": _safe_pct_diff(y, p),
            "abs_diff": y - p if pd.notna(y) and pd.notna(p) else np.nan,
        })

    return pd.DataFrame(rows)


# ============================================================
# 8) Batch comparison across many models
#    Ready for Long / Sign(R) / MACD / DQN / PG / A2C
# ============================================================

def compare_many_models_to_paper(
    results_by_model,
    asset_class,
):
    """
    results_by_model example:
    {
        "Long": {"table2_metrics": {...}, "table3_metrics": {...}},
        "Sign(R)": {...},
        "MACD": {...},
        "A2C": {...},
        ...
    }

    Returns:
        compare_t2_df, compare_t3_df
    """
    t2_frames = []
    t3_frames = []

    for model_name, res in results_by_model.items():
        if model_name in PAPER_TABLE2[asset_class]:
            t2_frames.append(
                compare_model_to_paper(
                    your_metrics=res["table2_metrics"],
                    paper_table=PAPER_TABLE2,
                    asset_class=asset_class,
                    model_name=model_name,
                )
            )

        if model_name in PAPER_TABLE3[asset_class]:
            t3_frames.append(
                compare_model_to_paper(
                    your_metrics=res["table3_metrics"],
                    paper_table=PAPER_TABLE3,
                    asset_class=asset_class,
                    model_name=model_name,
                )
            )

    compare_t2_df = pd.concat(t2_frames, ignore_index=True) if t2_frames else pd.DataFrame()
    compare_t3_df = pd.concat(t3_frames, ignore_index=True) if t3_frames else pd.DataFrame()

    return compare_t2_df, compare_t3_df


# ============================================================
# 9) Convenience helper:
#    build per-model per-asset-class results from raw pnl dicts
# ============================================================

def evaluate_models_for_asset_class(
    pnl_by_model_by_ticker,
    asset_class,
    periods_per_year=252,
    table2_target_vol=0.975,
    table2_ewm_span=60,
):
    """
    pnl_by_model_by_ticker example:
    {
        "Long":    {"ES": df, "TY": df, ...},
        "Sign(R)": {"ES": df, "TY": df, ...},
        "MACD":    {...},
        "A2C":     {...},
        "DQN":     {...},   # later
        "PG":      {...},   # later
    }

    Only keeps the tickers belonging to asset_class.
    """
    tickers = set(PAPER_TICKERS_BY_ASSET_CLASS[asset_class])
    results = {}

    for model_name, pnl_by_ticker in pnl_by_model_by_ticker.items():
        sub = {k: v for k, v in pnl_by_ticker.items() if k in tickers}
        if len(sub) == 0:
            continue

        results[model_name] = compute_model_table2_table3_results(
            sub,
            periods_per_year=periods_per_year,
            table2_target_vol=table2_target_vol,
            table2_ewm_span=table2_ewm_span,
        )

    return results


# ============================================================
# 10) Pretty summary table: metrics as columns
# ============================================================

def metrics_dict_to_frame(results_by_model, table_key="table2_metrics"):
    rows = []
    for model_name, res in results_by_model.items():
        row = {"model": model_name}
        row.update(res[table_key])
        rows.append(row)
    return pd.DataFrame(rows)


def paper_metrics_to_frame(paper_table, asset_class):
    rows = []
    for model_name, metrics in paper_table[asset_class].items():
        row = {"model": model_name}
        row.update(metrics)
        rows.append(row)
    return pd.DataFrame(rows)

In [9]:
import numpy as np
import pandas as pd
import torch

In [15]:
import torch

# 直接硬编码数据路径（到 data 目录，不是 CLC）这里是lyt自己的路径，大家记得改
#from google.colab import drive
#drive.mount('/content/drive')
#DATA_ROOT = "/content/drive/MyDrive/IEOR4733_Project/data"
import os
DATA_ROOT = "/Users/ladymie/Documents/GitHub/IEOR4733_Project/data"

STATE_FEATURES = [
    "close_norm",
    "ret_1m",
    "ret_2m",
    "ret_3m",
    "ret_1y",
    "macd_8_24",
    "macd_16_48",
    "macd_32_96",
    "rsi_30",
]

TRAIN_END_1 = "2010-12-31"
TEST_START_1 = "2011-01-01"
TEST_END_1 = "2015-12-31"

TRAIN_END_2 = "2015-12-31"
TEST_START_2 = "2016-01-01"
TEST_END_2 = "2019-12-31"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DATA_ROOT:", DATA_ROOT)
print("DEVICE:", DEVICE)

DATA_ROOT: /Users/ladymie/Documents/GitHub/IEOR4733_Project/data
DEVICE: cpu


In [16]:
data_dict, panel, missing = load_paper_rad_data(
    DATA_ROOT,
    start="2005-01-01",
    end="2019-12-31"
)

print("Loaded tickers:", len(data_dict))
print("Missing tickers:", missing)
print(panel.head())

Loaded tickers: 50
Missing tickers: []
        date    open    high     low   close  volume  open_interest ticker
0 2005-01-03  1848.0  1865.0  1831.0  1832.0    4759         117684     CC
1 2005-01-04  1830.0  1841.0  1821.0  1827.0    7117         118137     CC
2 2005-01-05  1821.0  1848.0  1820.0  1831.0    5350         118335     CC
3 2005-01-06  1824.0  1831.0  1807.0  1817.0    8218         117911     CC
4 2005-01-07  1809.0  1813.0  1764.0  1767.0   14843         119415     CC


In [17]:
feature_dict, feature_panel = build_paper_features(data_dict, dropna=False)

print(feature_panel.head())
print(feature_panel.columns.tolist())

        date ticker   open   high    low  close  volume  open_interest  \
0 2005-01-03     AN  59.27  59.33  58.47  59.00   14080          64957   
1 2005-01-04     AN  58.77  59.03  57.77  57.87   15875          66759   
2 2005-01-05     AN  57.91  58.15  57.49  57.92   12408          67640   
3 2005-01-06     AN  57.91  57.95  57.37  57.55   10195          66989   
4 2005-01-07     AN  57.68  58.12  57.10  57.23   15414          64094   

   ret_1d  ewm_vol_60  ...  ret_1y_raw  ret_1m  ret_2m  ret_3m  ret_1y  \
0     NaN         NaN  ...         NaN     NaN     NaN     NaN     NaN   
1   -1.13         NaN  ...         NaN     NaN     NaN     NaN     NaN   
2    0.05         NaN  ...         NaN     NaN     NaN     NaN     NaN   
3   -0.37         NaN  ...         NaN     NaN     NaN     NaN     NaN   
4   -0.32         NaN  ...         NaN     NaN     NaN     NaN     NaN   

   macd_8_24  macd_16_48  macd_32_96  macd_avg  rsi_30  
0        NaN         NaN         NaN       NaN     Na

In [18]:
pnl_long_by_ticker = {}
pnl_sign_by_ticker = {}
pnl_macd_by_ticker = {}

for ticker, df in feature_dict.items():
    x = df.copy()
    x = build_baselines(x)

    # 只保留测试期
    x_test = x[(x["date"] >= TEST_START_1) & (x["date"] <= TEST_END_2)].copy()

    long_pnl = compute_pnl(x_test, signal_col="long_only", dropna=True)
    sign_pnl = compute_pnl(x_test, signal_col="sign", dropna=True)
    macd_pnl = compute_pnl(x_test, signal_col="macd_signal", dropna=True)

    pnl_long_by_ticker[ticker] = long_pnl
    pnl_sign_by_ticker[ticker] = sign_pnl
    pnl_macd_by_ticker[ticker] = macd_pnl

print("Baseline pnl done.")

Baseline pnl done.


In [19]:
feature_train_1 = {}
feature_test_1 = {}
feature_train_2 = {}
feature_test_2 = {}

for ticker, df in feature_dict.items():
    feature_train_1[ticker] = df[df["date"] <= TRAIN_END_1].copy()
    feature_test_1[ticker]  = df[(df["date"] >= TEST_START_1) & (df["date"] <= TEST_END_1)].copy()

    feature_train_2[ticker] = df[df["date"] <= TRAIN_END_2].copy()
    feature_test_2[ticker]  = df[(df["date"] >= TEST_START_2) & (df["date"] <= TEST_END_2)].copy()

print("Train/test split done.")

Train/test split done.


In [20]:
state_train_1 = make_state_tensors_all(feature_train_1, feature_cols=STATE_FEATURES, window=60)
state_test_1  = make_state_tensors_all(feature_test_1,  feature_cols=STATE_FEATURES, window=60)

state_train_2 = make_state_tensors_all(feature_train_2, feature_cols=STATE_FEATURES, window=60)
state_test_2  = make_state_tensors_all(feature_test_2,  feature_cols=STATE_FEATURES, window=60)

print("State tensors built.")

State tensors built.


In [21]:
a2c_models_period1 = {}

for asset_class in ["Commodity", "Equity Index", "Fixed Income", "FX"]:
    tickers = PAPER_TICKERS_BY_ASSET_CLASS[asset_class]

    valid_tickers = []
    for ticker in tickers:
        if ticker not in state_train_1:
            continue

        X = state_train_1[ticker]["X"]
        aligned_df = state_train_1[ticker]["aligned_df"]

        if not isinstance(aligned_df, pd.DataFrame):
            continue
        if len(aligned_df) == 0 or len(X) == 0:
            continue
        if not all(c in aligned_df.columns for c in ["close", "ret_1d", "ewm_vol_60"]):
            continue

        valid_tickers.append(ticker)

    print(asset_class, "valid tickers (period 1):", valid_tickers)

    envs = build_envs_from_state_dict(
        state_train_1,
        tickers=valid_tickers,
        sigma_target=0.064,   # 你现在实验里用的实现假设
        bp=BP,
        mu=MU,
    )

    if len(envs) == 0:
        print(f"{asset_class}: no envs in period 1")
        continue

    # ⭐ 关键改动：让有效 batch size 接近论文 Table 1 的 128
    rollout_steps = max(1, round(BATCH_SIZE / len(envs)))
    effective_batch = rollout_steps * len(envs)

    print(
        f"{asset_class}: envs={len(envs)}, "
        f"rollout_steps={rollout_steps}, "
        f"effective_batch={effective_batch}"
    )

    trainer = PaperA2CTrainer(
        n_features=len(STATE_FEATURES),
        device=DEVICE
    )

    ckpt_path = f"a2c_{asset_class}_period1.pt"

    start_update = 0
    if os.path.exists(ckpt_path):
        ckpt = trainer.load_checkpoint(ckpt_path, map_location=DEVICE)
        start_update = ckpt.get("extra_state", {}).get("update_idx", 0)
        print(f"resumed from {ckpt_path}, start_update={start_update}")

    trainer.fit(
        envs=envs,
        n_updates=100,                 # 每次先续 100
        rollout_steps=rollout_steps,   # ⭐ 这里不再写死 32
        log_every=20,
        checkpoint_path=ckpt_path,
        checkpoint_every=20,
        start_update=start_update,
    )

    a2c_models_period1[asset_class] = trainer

print("A2C period 1 training done.")

Commodity valid tickers (period 1): ['CC', 'DA', 'GI', 'JO', 'KC', 'KW', 'LB', 'NR', 'SB', 'ZA', 'ZC', 'ZF', 'ZG', 'ZI', 'ZK', 'ZL', 'ZO', 'ZP', 'ZR', 'ZT', 'ZW', 'ZZ']
Commodity: envs=22, rollout_steps=6, effective_batch=132
resumed from a2c_Commodity_period1.pt, start_update=2000
[2020] actor_loss=-0.1542 critic_loss=0.0018 avg_reward=0.005534 entropy=0.8600
checkpoint saved to a2c_Commodity_period1.pt
[2040] actor_loss=0.0136 critic_loss=0.0022 avg_reward=-0.002261 entropy=0.8592
checkpoint saved to a2c_Commodity_period1.pt
[2060] actor_loss=0.0454 critic_loss=0.0030 avg_reward=-0.006722 entropy=0.8583
checkpoint saved to a2c_Commodity_period1.pt
[2080] actor_loss=0.0062 critic_loss=0.0019 avg_reward=-0.009851 entropy=0.8573
checkpoint saved to a2c_Commodity_period1.pt
[2100] actor_loss=-0.0375 critic_loss=0.0023 avg_reward=0.004546 entropy=0.8570
checkpoint saved to a2c_Commodity_period1.pt
Equity Index valid tickers (period 1): ['CA', 'EN', 'ER', 'ES', 'LX', 'MD', 'SC', 'SP', 'XU'

In [22]:
a2c_models_period1_all = None

# 训练 ALL 资产类的 A2C
tickers_all = PAPER_TICKERS_BY_ASSET_CLASS["All"]

valid_tickers_all = []
for ticker in tickers_all:
    if ticker not in state_train_1:
        continue

    X = state_train_1[ticker]["X"]
    aligned_df = state_train_1[ticker]["aligned_df"]

    if not isinstance(aligned_df, pd.DataFrame):
        continue
    if len(aligned_df) == 0 or len(X) == 0:
        continue
    if not all(c in aligned_df.columns for c in ["close", "ret_1d", "ewm_vol_60"]):
        continue

    valid_tickers_all.append(ticker)

print("ALL valid tickers (period 1):", len(valid_tickers_all))

envs_all = build_envs_from_state_dict(
    state_train_1,
    tickers=valid_tickers_all,
    sigma_target=0.064,
    bp=BP,
    mu=MU,
)

if len(envs_all) > 0:
    rollout_steps_all = max(1, round(BATCH_SIZE / len(envs_all)))
    effective_batch_all = rollout_steps_all * len(envs_all)

    print(
        f"ALL: envs={len(envs_all)}, "
        f"rollout_steps={rollout_steps_all}, "
        f"effective_batch={effective_batch_all}"
    )

    trainer_all = PaperA2CTrainer(
        n_features=len(STATE_FEATURES),
        device=DEVICE
    )

    # 使用相对路径，这样在 Colab 中会保存到当前工作目录 (/content)
    ckpt_path_all = "a2c_All_period1.pt"

    start_update = 0
    if os.path.exists(ckpt_path_all):
        ckpt = trainer_all.load_checkpoint(ckpt_path_all, map_location=DEVICE)
        start_update = ckpt.get("extra_state", {}).get("update_idx", 0)
        print(f"resumed from {ckpt_path_all}, start_update={start_update}")

    trainer_all.fit(
        envs=envs_all,
        n_updates=100,
        rollout_steps=rollout_steps_all,
        log_every=20,
        checkpoint_path=ckpt_path_all,
        checkpoint_every=20,
        start_update=start_update,
    )

    a2c_models_period1_all = trainer_all

print("A2C period 1 (ALL) training done.")

ALL valid tickers (period 1): 46
ALL: envs=46, rollout_steps=3, effective_batch=138
resumed from a2c_All_period1.pt, start_update=100
[ 120] actor_loss=-0.0016 critic_loss=0.0011 avg_reward=-0.009490 entropy=0.9117
checkpoint saved to a2c_All_period1.pt
[ 140] actor_loss=0.0004 critic_loss=0.0009 avg_reward=-0.016090 entropy=0.9105
checkpoint saved to a2c_All_period1.pt
[ 160] actor_loss=0.0468 critic_loss=0.0011 avg_reward=-0.009367 entropy=0.9089
checkpoint saved to a2c_All_period1.pt
[ 180] actor_loss=-0.0071 critic_loss=0.0016 avg_reward=-0.011745 entropy=0.9075
checkpoint saved to a2c_All_period1.pt
[ 200] actor_loss=-0.0814 critic_loss=0.0015 avg_reward=-0.013576 entropy=0.9068
checkpoint saved to a2c_All_period1.pt
A2C period 1 (ALL) training done.


In [23]:
pnl_a2c_by_ticker_period1 = {}

for asset_class in ["Commodity", "Equity Index", "Fixed Income", "FX"]:
    if asset_class not in a2c_models_period1:
        continue

    trainer = a2c_models_period1[asset_class]
    tickers = PAPER_TICKERS_BY_ASSET_CLASS[asset_class]

    for ticker in tickers:
        if ticker not in state_test_1:
            continue

        X = state_test_1[ticker]["X"]
        aligned_df = state_test_1[ticker]["aligned_df"].copy()

        if len(X) == 0 or len(aligned_df) == 0:
            continue

        with torch.no_grad():
            x_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
            actions = trainer.actor.deterministic_action(x_tensor).squeeze(-1).cpu().numpy()

        aligned_df["a2c_signal"] = actions
        pnl_df = compute_pnl(aligned_df, signal_col="a2c_signal", dropna=True)

        pnl_a2c_by_ticker_period1[ticker] = pnl_df

print("A2C period 1 test pnl done.")

A2C period 1 test pnl done.


In [24]:
if a2c_models_period1_all is not None:
    trainer_all = a2c_models_period1_all
    tickers_all = PAPER_TICKERS_BY_ASSET_CLASS["All"]

    for ticker in tickers_all:
        if ticker not in state_test_1:
            continue

        X = state_test_1[ticker]["X"]
        aligned_df = state_test_1[ticker]["aligned_df"].copy()

        if len(X) == 0 or len(aligned_df) == 0:
            continue

        with torch.no_grad():
            x_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
            actions = trainer_all.actor.deterministic_action(x_tensor).squeeze(-1).cpu().numpy()

        aligned_df["a2c_signal"] = actions
        pnl_df = compute_pnl(aligned_df, signal_col="a2c_signal", dropna=True)

        pnl_a2c_by_ticker_period1[ticker] = pnl_df

print("A2C period 1 (ALL) test pnl done.")

A2C period 1 (ALL) test pnl done.


In [25]:
a2c_models_period2 = {}

for asset_class in ["Commodity", "Equity Index", "Fixed Income", "FX"]:
    tickers = PAPER_TICKERS_BY_ASSET_CLASS[asset_class]

    valid_tickers = []
    for ticker in tickers:
        if ticker not in state_train_2:
            continue

        X = state_train_2[ticker]["X"]
        aligned_df = state_train_2[ticker]["aligned_df"]

        if not isinstance(aligned_df, pd.DataFrame):
            continue
        if len(aligned_df) == 0 or len(X) == 0:
            continue
        if not all(c in aligned_df.columns for c in ["close", "ret_1d", "ewm_vol_60"]):
            continue

        valid_tickers.append(ticker)

    print(asset_class, "valid tickers (period 2):", valid_tickers)

    envs = build_envs_from_state_dict(
        state_train_2,
        tickers=valid_tickers,
        sigma_target=0.064,
        bp=BP,
        mu=MU,
    )

    if len(envs) == 0:
        print(f"{asset_class}: no envs in period 2")
        continue

    rollout_steps = max(1, round(BATCH_SIZE / len(envs)))
    effective_batch = rollout_steps * len(envs)

    print(
        f"{asset_class}: envs={len(envs)}, "
        f"rollout_steps={rollout_steps}, "
        f"effective_batch={effective_batch}"
    )

    trainer = PaperA2CTrainer(
        n_features=len(STATE_FEATURES),
        device=DEVICE
    )

    ckpt_path = f"a2c_{asset_class}_period2.pt"

    start_update = 0
    if os.path.exists(ckpt_path):
        ckpt = trainer.load_checkpoint(ckpt_path, map_location=DEVICE)
        start_update = ckpt.get("extra_state", {}).get("update_idx", 0)
        print(f"resumed from {ckpt_path}, start_update={start_update}")

    trainer.fit(
        envs=envs,
        n_updates=100,
        rollout_steps=rollout_steps,
        log_every=20,
        checkpoint_path=ckpt_path,
        checkpoint_every=20,
        start_update=start_update,
    )

    a2c_models_period2[asset_class] = trainer

print("A2C period 2 training done.")

Commodity valid tickers (period 2): ['CC', 'DA', 'GI', 'JO', 'KC', 'KW', 'LB', 'NR', 'SB', 'ZA', 'ZC', 'ZF', 'ZG', 'ZI', 'ZK', 'ZL', 'ZN', 'ZO', 'ZP', 'ZR', 'ZT', 'ZW', 'ZZ']
Commodity: envs=23, rollout_steps=6, effective_batch=138
resumed from a2c_Commodity_period2.pt, start_update=2000
[2020] actor_loss=-0.1452 critic_loss=0.0015 avg_reward=0.003767 entropy=0.8654
checkpoint saved to a2c_Commodity_period2.pt
[2040] actor_loss=0.0389 critic_loss=0.0025 avg_reward=-0.004788 entropy=0.8651
checkpoint saved to a2c_Commodity_period2.pt
[2060] actor_loss=-0.0714 critic_loss=0.0039 avg_reward=-0.002157 entropy=0.8646
checkpoint saved to a2c_Commodity_period2.pt
[2080] actor_loss=0.0115 critic_loss=0.0021 avg_reward=-0.011507 entropy=0.8638
checkpoint saved to a2c_Commodity_period2.pt
[2100] actor_loss=-0.1126 critic_loss=0.0022 avg_reward=0.000770 entropy=0.8642
checkpoint saved to a2c_Commodity_period2.pt
Equity Index valid tickers (period 2): ['CA', 'EN', 'ER', 'ES', 'LX', 'MD', 'SC', 'SP

In [26]:
a2c_models_period2_all = None

# 训练 ALL 资产类的 A2C (period 2)
tickers_all = PAPER_TICKERS_BY_ASSET_CLASS["All"]

valid_tickers_all = []
for ticker in tickers_all:
    if ticker not in state_train_2:
        continue

    X = state_train_2[ticker]["X"]
    aligned_df = state_train_2[ticker]["aligned_df"]

    if not isinstance(aligned_df, pd.DataFrame):
        continue
    if len(aligned_df) == 0 or len(X) == 0:
        continue
    if not all(c in aligned_df.columns for c in ["close", "ret_1d", "ewm_vol_60"]):
        continue

    valid_tickers_all.append(ticker)

print("ALL valid tickers (period 2):", len(valid_tickers_all))

envs_all = build_envs_from_state_dict(
    state_train_2,
    tickers=valid_tickers_all,
    sigma_target=0.064,
    bp=BP,
    mu=MU,
)

if len(envs_all) > 0:
    rollout_steps_all = max(1, round(BATCH_SIZE / len(envs_all)))
    effective_batch_all = rollout_steps_all * len(envs_all)

    print(
        f"ALL: envs={len(envs_all)}, "
        f"rollout_steps={rollout_steps_all}, "
        f"effective_batch={effective_batch_all}"
    )

    trainer_all = PaperA2CTrainer(
        n_features=len(STATE_FEATURES),
        device=DEVICE
    )

    # 使用绝对路径确保文件保存到 rl_models 目录
    ckpt_path_all = "/Users/ladymie/Documents/GitHub/IEOR4733_Project/rl_models/a2c_All_period2.pt"
    
    # 确保父目录存在
    os.makedirs(os.path.dirname(ckpt_path_all), exist_ok=True)

    start_update = 0
    if os.path.exists(ckpt_path_all):
        ckpt = trainer_all.load_checkpoint(ckpt_path_all, map_location=DEVICE)
        start_update = ckpt.get("extra_state", {}).get("update_idx", 0)
        print(f"resumed from {ckpt_path_all}, start_update={start_update}")

    trainer_all.fit(
        envs=envs_all,
        n_updates=100,
        rollout_steps=rollout_steps_all,
        log_every=20,
        checkpoint_path=ckpt_path_all,
        checkpoint_every=20,
        start_update=start_update,
    )

    a2c_models_period2_all = trainer_all

ALL valid tickers (period 2): 47
ALL: envs=47, rollout_steps=3, effective_batch=141
resumed from /Users/ladymie/Documents/GitHub/IEOR4733_Project/rl_models/a2c_All_period2.pt, start_update=100
[ 120] actor_loss=-0.0767 critic_loss=0.0015 avg_reward=-0.014519 entropy=0.9120
checkpoint saved to /Users/ladymie/Documents/GitHub/IEOR4733_Project/rl_models/a2c_All_period2.pt
[ 140] actor_loss=-0.0643 critic_loss=0.0009 avg_reward=-0.007924 entropy=0.9111
checkpoint saved to /Users/ladymie/Documents/GitHub/IEOR4733_Project/rl_models/a2c_All_period2.pt
[ 160] actor_loss=-0.0016 critic_loss=0.0008 avg_reward=-0.010668 entropy=0.9101
checkpoint saved to /Users/ladymie/Documents/GitHub/IEOR4733_Project/rl_models/a2c_All_period2.pt
[ 180] actor_loss=0.0342 critic_loss=0.0025 avg_reward=-0.010079 entropy=0.9090
checkpoint saved to /Users/ladymie/Documents/GitHub/IEOR4733_Project/rl_models/a2c_All_period2.pt
[ 200] actor_loss=-0.0706 critic_loss=0.0017 avg_reward=-0.005402 entropy=0.9083
checkpoint 

In [27]:
pnl_a2c_by_ticker_period2 = {}

for asset_class in ["Commodity", "Equity Index", "Fixed Income", "FX"]:
    if asset_class not in a2c_models_period2:
        continue

    trainer = a2c_models_period2[asset_class]
    tickers = PAPER_TICKERS_BY_ASSET_CLASS[asset_class]

    for ticker in tickers:
        if ticker not in state_test_2:
            continue

        X = state_test_2[ticker]["X"]
        aligned_df = state_test_2[ticker]["aligned_df"].copy()

        if len(X) == 0 or len(aligned_df) == 0:
            continue

        with torch.no_grad():
            x_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
            actions = trainer.actor.deterministic_action(x_tensor).squeeze(-1).cpu().numpy()

        aligned_df["a2c_signal"] = actions
        pnl_df = compute_pnl(aligned_df, signal_col="a2c_signal", dropna=True)

        pnl_a2c_by_ticker_period2[ticker] = pnl_df

print("A2C period 2 test pnl done.")

A2C period 2 test pnl done.


In [28]:
if a2c_models_period2_all is not None:
    trainer_all = a2c_models_period2_all
    tickers_all = PAPER_TICKERS_BY_ASSET_CLASS["All"]

    for ticker in tickers_all:
        if ticker not in state_test_2:
            continue

        X = state_test_2[ticker]["X"]
        aligned_df = state_test_2[ticker]["aligned_df"].copy()

        if len(X) == 0 or len(aligned_df) == 0:
            continue

        with torch.no_grad():
            x_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
            actions = trainer_all.actor.deterministic_action(x_tensor).squeeze(-1).cpu().numpy()

        aligned_df["a2c_signal"] = actions
        pnl_df = compute_pnl(aligned_df, signal_col="a2c_signal", dropna=True)

        pnl_a2c_by_ticker_period2[ticker] = pnl_df

print("A2C period 2 (ALL) test pnl done.")

A2C period 2 (ALL) test pnl done.


In [30]:
pnl_a2c_by_ticker = {}

all_tickers = sorted(set(list(pnl_a2c_by_ticker_period1.keys()) + list(pnl_a2c_by_ticker_period2.keys())))

for ticker in all_tickers:
    parts = []
    if ticker in pnl_a2c_by_ticker_period1:
        parts.append(pnl_a2c_by_ticker_period1[ticker])
    if ticker in pnl_a2c_by_ticker_period2:
        parts.append(pnl_a2c_by_ticker_period2[ticker])

    if len(parts) > 0:
        pnl_a2c_by_ticker[ticker] = (
            pd.concat(parts, ignore_index=True)
            .sort_values("date")
            .reset_index(drop=True)
        )

print("A2C full test pnl done.")

A2C full test pnl done.


In [31]:
pnl_by_model_by_ticker = {
    "Long": pnl_long_by_ticker,
    "Sign(R)": pnl_sign_by_ticker,
    "MACD": pnl_macd_by_ticker,
    "A2C": pnl_a2c_by_ticker,
}

In [32]:
results_all = evaluate_models_for_asset_class(
    pnl_by_model_by_ticker,
    asset_class="All",
    table2_target_vol=0.975,
    table2_ewm_span=60
)

my_table2_all = metrics_dict_to_frame(results_all, table_key="table2_metrics")
my_table3_all = metrics_dict_to_frame(results_all, table_key="table3_metrics")

compare_t2_all, compare_t3_all = compare_many_models_to_paper(
    results_all,
    asset_class="All"
)

print("=== YOUR TABLE 2 STYLE RESULTS: ALL ===")
display(my_table2_all)

print("=== YOUR TABLE 3 STYLE RESULTS: ALL ===")
display(my_table3_all)

print("=== COMPARE VS PAPER TABLE 2: ALL ===")
display(compare_t2_all)

print("=== COMPARE VS PAPER TABLE 3: ALL ===")
display(compare_t3_all)

=== YOUR TABLE 2 STYLE RESULTS: ALL ===


,model,E(R),Std(R),DD,Sharpe,Sortino,MDD,Calmar,% +Ret,Ave.P/Ave.L
0,Long,0.144603,0.994800,0.699450,0.145359,0.206739,3.936406,0.036735,0.520723,0.942730
1,Sign(R),-0.441257,0.999741,0.702900,-0.441371,-0.627766,5.630114,-0.078374,0.499559,0.929839
2,MACD,-1.160711,0.994988,0.691106,-1.166558,-1.679498,11.204944,-0.103589,0.468254,0.932236
3,A2C,0.114342,0.991113,0.626328,0.115368,0.182560,4.103775,0.027863,0.501864,1.011575


=== YOUR TABLE 3 STYLE RESULTS: ALL ===


,model,E(R),Std(R),DD,Sharpe,Sortino,MDD,Calmar,% +Ret,Ave.P/Ave.L
0,Long,0.065299,0.369977,0.261084,0.176494,0.250106,1.301283,0.050180,0.521048,0.946413
1,Sign(R),-0.119033,0.307186,0.220087,-0.387496,-0.540846,1.639335,-0.072611,0.500859,0.932970
2,MACD,-0.308164,0.251335,0.178671,-1.226108,-1.724757,3.164110,-0.097394,0.468643,0.919022
3,A2C,-0.000205,0.018085,0.011361,-0.011348,-0.018064,0.080982,-0.002534,0.499093,1.001760


=== COMPARE VS PAPER TABLE 2: ALL ===


,asset_class,model,metric,your_value,paper_value,pct_diff_vs_paper,abs_diff
0,All,Long,E(R),0.144603,0.055,162.915205,0.089603
1,All,Long,Std(R),0.994800,0.975,2.030817,0.019800
2,All,Long,DD,0.699450,0.598,16.964863,0.101450
3,All,Long,Sharpe,0.145359,0.058,150.619245,0.087359
4,All,Long,Sortino,0.206739,0.092,124.715984,0.114739
5,All,Long,MDD,3.936406,0.071,5444.233734,3.865406
6,All,Long,Calmar,0.036735,0.013,182.575927,0.023735
7,All,Long,% +Ret,0.520723,0.520,0.139058,0.000723
8,All,Long,Ave.P/Ave.L,0.942730,0.933,1.042874,0.009730
9,All,Sign(R),E(R),-0.441257,0.429,-202.857025,-0.870257


=== COMPARE VS PAPER TABLE 3: ALL ===


,asset_class,model,metric,your_value,paper_value,pct_diff_vs_paper,abs_diff
0,All,Long,E(R),0.065299,-0.013,602.297418,0.078299
1,All,Long,Std(R),0.369977,0.363,1.921977,0.006977
2,All,Long,DD,0.261084,0.230,13.514915,0.031084
3,All,Long,Sharpe,0.176494,-0.036,590.260987,0.212494
4,All,Long,Sortino,0.250106,-0.057,538.781875,0.307106
5,All,Long,MDD,1.301283,0.037,3416.981022,1.264283
6,All,Long,Calmar,0.050180,-0.009,657.557985,0.059180
7,All,Long,% +Ret,0.521048,0.519,0.394626,0.002048
8,All,Long,Ave.P/Ave.L,0.946413,0.919,2.982897,0.027413
9,All,Sign(R),E(R),-0.119033,0.086,-238.410693,-0.205033


In [33]:
all_asset_classes = ["Commodity", "Equity Index", "Fixed Income", "FX", "All"]

all_compare_t2 = []
all_compare_t3 = []
all_my_t2 = []
all_my_t3 = []

for asset_class in all_asset_classes:
    results = evaluate_models_for_asset_class(
        pnl_by_model_by_ticker,
        asset_class=asset_class,
        table2_target_vol=0.975,
        table2_ewm_span=60
    )

    my_t2 = metrics_dict_to_frame(results, table_key="table2_metrics")
    my_t2["asset_class"] = asset_class
    all_my_t2.append(my_t2)

    my_t3 = metrics_dict_to_frame(results, table_key="table3_metrics")
    my_t3["asset_class"] = asset_class
    all_my_t3.append(my_t3)

    c2, c3 = compare_many_models_to_paper(results, asset_class=asset_class)

    if not c2.empty:
        all_compare_t2.append(c2)
    if not c3.empty:
        all_compare_t3.append(c3)

my_table2_full = pd.concat(all_my_t2, ignore_index=True)
my_table3_full = pd.concat(all_my_t3, ignore_index=True)
compare_t2_full = pd.concat(all_compare_t2, ignore_index=True)
compare_t3_full = pd.concat(all_compare_t3, ignore_index=True)

print("=== YOUR TABLE 2 STYLE RESULTS: FULL ===")
display(my_table2_full)

print("=== YOUR TABLE 3 STYLE RESULTS: FULL ===")
display(my_table3_full)

print("=== COMPARE VS PAPER TABLE 2: FULL ===")
display(compare_t2_full)

print("=== COMPARE VS PAPER TABLE 3: FULL ===")
display(compare_t3_full)

=== YOUR TABLE 2 STYLE RESULTS: FULL ===


,model,E(R),Std(R),DD,Sharpe,Sortino,MDD,Calmar,% +Ret,Ave.P/Ave.L,asset_class
0,Long,-0.505794,0.988526,0.655722,-0.511665,-0.771354,6.332914,-0.079868,0.498866,0.924052,Commodity
1,Sign(R),-0.542711,0.988617,0.631013,-0.548960,-0.860064,6.089412,-0.089124,0.487075,0.963480,Commodity
2,MACD,-1.074184,0.990973,0.640377,-1.083970,-1.677425,9.894272,-0.108566,0.473469,0.930780,Commodity
3,A2C,0.536742,0.992067,0.599550,0.541034,0.895241,2.623418,0.204596,0.513202,1.035733,Commodity
4,Long,0.562991,1.004456,0.789714,0.560493,0.712905,1.808170,0.311359,0.547788,0.908991,Equity Index
5,Sign(R),0.038283,1.016285,0.766896,0.037669,0.049919,3.150030,0.012153,0.517257,0.939537,Equity Index
6,MACD,-0.447639,1.009037,0.794942,-0.443630,-0.563109,4.878821,-0.091752,0.506195,0.900217,Equity Index
7,A2C,0.096678,1.004396,0.692362,0.096254,0.139634,4.110128,0.023522,0.499532,1.018619,Equity Index
8,Long,0.563561,1.000004,0.666671,0.563559,0.845337,2.280055,0.247170,0.525011,0.992554,Fixed Income
9,Sign(R),-0.289575,1.004333,0.714171,-0.288326,-0.405471,5.489288,-0.052753,0.513059,0.904080,Fixed Income


=== YOUR TABLE 3 STYLE RESULTS: FULL ===


,model,E(R),Std(R),DD,Sharpe,Sortino,MDD,Calmar,% +Ret,Ave.P/Ave.L,asset_class
0,Long,-0.194787,0.403858,0.274291,-0.482317,-0.710148,2.634746,-0.073930,0.501104,0.919846,Commodity
1,Sign(R),-0.147929,0.310864,0.205308,-0.475865,-0.720526,1.783202,-0.082957,0.489183,0.965871,Commodity
2,MACD,-0.255898,0.238791,0.155663,-1.071640,-1.643923,2.594337,-0.098637,0.475055,0.925910,Commodity
3,A2C,0.010514,0.019010,0.011294,0.553072,0.930920,0.042884,0.245178,0.513299,1.037756,Commodity
4,Long,0.581131,0.926723,0.727137,0.627081,0.799203,1.781641,0.326177,0.547845,0.918653,Equity Index
5,Sign(R),0.060224,0.796312,0.632154,0.075629,0.095268,2.472464,0.024358,0.518103,0.942736,Equity Index
6,MACD,-0.266897,0.615344,0.481559,-0.433737,-0.554236,3.283955,-0.081273,0.506034,0.902567,Equity Index
7,A2C,0.001292,0.047689,0.031644,0.027095,0.040833,0.229969,0.005619,0.498635,1.010153,Equity Index
8,Long,0.516827,0.946275,0.616012,0.546170,0.838988,2.118845,0.243919,0.525658,0.986335,Fixed Income
9,Sign(R),-0.075754,0.826930,0.578990,-0.091608,-0.130837,3.725731,-0.020333,0.514015,0.930857,Fixed Income


=== COMPARE VS PAPER TABLE 2: FULL ===


,asset_class,model,metric,your_value,paper_value,pct_diff_vs_paper,abs_diff
0,Commodity,Long,E(R),-0.505794,-0.710,28.761386,0.204206
1,Commodity,Long,Std(R),0.988526,0.979,0.973012,0.009526
2,Commodity,Long,DD,0.655722,0.604,8.563288,0.051722
3,Commodity,Long,Sharpe,-0.511665,-0.726,29.522711,0.214335
4,Commodity,Long,Sortino,-0.771354,-1.177,34.464378,0.405646
...,...,...,...,...,...,...,...
175,All,A2C,Sortino,0.182560,1.785,-89.772559,-1.602440
176,All,A2C,MDD,4.103775,0.007,58525.356268,4.096775
177,All,A2C,Calmar,0.027863,0.685,-95.932449,-0.657137
178,All,A2C,% +Ret,0.501864,0.538,-6.716741,-0.036136


=== COMPARE VS PAPER TABLE 3: FULL ===


,asset_class,model,metric,your_value,paper_value,pct_diff_vs_paper,abs_diff
0,Commodity,Long,E(R),-0.194787,-0.298,34.635153,0.103213
1,Commodity,Long,Std(R),0.403858,0.412,-1.976310,-0.008142
2,Commodity,Long,DD,0.274291,0.258,6.314339,0.016291
3,Commodity,Long,Sharpe,-0.482317,-0.723,33.289537,0.240683
4,Commodity,Long,Sortino,-0.710148,-1.152,38.355195,0.441852
...,...,...,...,...,...,...,...
175,All,A2C,Sortino,-0.018064,1.601,-101.128274,-1.619064
176,All,A2C,MDD,0.080982,0.009,799.801431,0.071982
177,All,A2C,Calmar,-0.002534,0.672,-100.377101,-0.674534
178,All,A2C,% +Ret,0.499093,0.538,-7.231713,-0.038907


In [34]:
def build_a2c_results_by_asset(pnl_a2c_by_ticker, ticker_to_asset):
    results = {}
    asset_classes = ['Commodity', 'Equity Index', 'Fixed Income', 'Forex']

    for asset in asset_classes:
        series_list = []

        for ticker, df in pnl_a2c_by_ticker.items():
            if ticker_to_asset.get(ticker) != asset:
                continue
            if df is None or len(df) == 0:
                continue
            if not all(c in df.columns for c in ['date', 'net_pnl']):
                continue

            s = pd.Series(df['net_pnl'].values, index=pd.to_datetime(df['date']))
            series_list.append(s)

        if series_list:
            port_series = pd.DataFrame(series_list).T.sort_index().mean(axis=1)
            dates = port_series.index.to_numpy()
            wealth = 1.0 + np.cumsum(port_series.values)
            results[asset] = (dates, wealth)
        else:
            results[asset] = (None, None)

    # All
    all_series = []
    for asset in asset_classes:
        dates, wealth = results[asset]
        if dates is not None:
            pass

    all_ticker_series = []
    for ticker, df in pnl_a2c_by_ticker.items():
        if df is None or len(df) == 0:
            continue
        if not all(c in df.columns for c in ['date', 'net_pnl']):
            continue
        s = pd.Series(df['net_pnl'].values, index=pd.to_datetime(df['date']))
        all_ticker_series.append(s)

    if all_ticker_series:
        port_series_all = pd.DataFrame(all_ticker_series).T.sort_index().mean(axis=1)
        results['All'] = (
            port_series_all.index.to_numpy(),
            1.0 + np.cumsum(port_series_all.values)
        )
    else:
        results['All'] = (None, None)

    return results

In [ ]:
# === 导出所有结果为 CSV 格式，供 strategies_comparison.ipynb 使用 ===

import os
from pathlib import Path
import pandas as pd

# 首先检查必要的变量是否存在
print("=" * 80)
print("检查必要数据...")
print("=" * 80)

required_vars = {
    'my_table2_full': '所有资产类别的 Table 2 结果',
    'my_table3_full': '所有资产类别的 Table 3 结果',
    'pnl_by_model_by_ticker': 'PnL 数据',
    'PAPER_TICKERS_BY_ASSET_CLASS': 'Ticker 分类'
}

missing_vars = []
for var_name, description in required_vars.items():
    if var_name in globals():
        print(f"✅ {var_name:30s}: 已定义")
    else:
        print(f"❌ {var_name:30s}: 未定义 - {description}")
        missing_vars.append(var_name)

if missing_vars:
    print("\n" + "=" * 80)
    print("⚠️  缺少必要的数据变量!")
    print("=" * 80)
    print("\n请按顺序执行以下单元格来生成这些数据:")
    print("  1. 第 27 个单元格: 生成 pnl_by_model_by_ticker")
    print("  2. 第 28 个单元格: 生成 my_table2_full, my_table3_full, compare_t2_full, compare_t3_full")
    print("\n然后重新运行此单元格")
    raise NameError(f"缺少必要的变量: {', '.join(missing_vars)}")

print("\n" + "=" * 80)
print("导出所有结果为 CSV 格式...")
print("=" * 80)

# 检查运行环境（Colab vs 本地）
try:
    from google.colab import files
    IN_COLAB = True
    # 保存到 Google Drive 中的项目文件夹
    output_dir = Path('/content/drive/MyDrive/IEOR4733_Project/reproduction_of_figures')
    print("📍 Colab 环境 - 保存到 Google Drive")
except ImportError:
    IN_COLAB = False
    print("📍 本地环境")
    # 本地环境保存到指定目录
    output_dir = Path('/Users/ladymie/Documents/GitHub/IEOR4733_Project/reproduction_of_figures')

output_dir.mkdir(parents=True, exist_ok=True)
print(f"📂 输出目录: {output_dir}\n")

try:
    exported_files = []
    
    # 导出 Table 2 结果
    t2_path = output_dir / 'a2c_all_results_table2.csv'
    my_table2_full.to_csv(t2_path, index=False)
    print(f"✅ Table 2 已保存: {t2_path.name}")
    exported_files.append(('a2c_all_results_table2.csv', t2_path))
    
    # 导出 Table 3 结果
    t3_path = output_dir / 'a2c_all_results_table3.csv'
    my_table3_full.to_csv(t3_path, index=False)
    print(f"✅ Table 3 已保存: {t3_path.name}")
    exported_files.append(('a2c_all_results_table3.csv', t3_path))
    
    # 导出与论文的对比 - Table 2
    if 'compare_t2_full' in globals() and not compare_t2_full.empty:
        cmp_t2_path = output_dir / 'a2c_compare_vs_paper_table2.csv'
        compare_t2_full.to_csv(cmp_t2_path, index=False)
        print(f"✅ Compare Table 2 已保存: {cmp_t2_path.name}")
        exported_files.append(('a2c_compare_vs_paper_table2.csv', cmp_t2_path))
    
    # 导出与论文的对比 - Table 3
    if 'compare_t3_full' in globals() and not compare_t3_full.empty:
        cmp_t3_path = output_dir / 'a2c_compare_vs_paper_table3.csv'
        compare_t3_full.to_csv(cmp_t3_path, index=False)
        print(f"✅ Compare Table 3 已保存: {cmp_t3_path.name}")
        exported_files.append(('a2c_compare_vs_paper_table3.csv', cmp_t3_path))
    
    # 导出 PnL 数据（用于画曲线）
    print("\n导出 PnL 数据...")
    
    pnl_count = 0
    
    # 对每个资产类别和每个策略导出组合 PnL
    for asset_class in ["Commodity", "Equity Index", "Fixed Income", "FX", "All"]:
        for model_name in ["Long", "Sign(R)", "MACD", "A2C"]:
            pnl_data = pnl_by_model_by_ticker.get(model_name, {})
            
            if not pnl_data:
                continue
            
            # 筛选该资产类别的 ticker
            tickers = PAPER_TICKERS_BY_ASSET_CLASS.get(asset_class, [])
            asset_pnls = []
            
            for ticker in tickers:
                if ticker in pnl_data:
                    pnl_df = pnl_data[ticker].copy()
                    # 确保有 date 列
                    if 'date' not in pnl_df.columns:
                        continue
                    # 使用 net_pnl 作为收益指标（已经是每日的 PnL）
                    asset_pnls.append(pnl_df[['date', 'net_pnl']].copy())
            
            if asset_pnls:
                # 合并所有 ticker 的 PnL
                combined = pd.concat(asset_pnls, ignore_index=True)
                # 按日期分组，计算平均 net_pnl（即平均日收益）
                daily_stats = combined.groupby('date').agg({'net_pnl': 'mean'}).reset_index()
                daily_stats.rename(columns={'net_pnl': 'daily_pnl'}, inplace=True)
                
                # 计算累积财富（假设初始财富为 1）
                daily_stats['cum_wealth'] = (1 + daily_stats['daily_pnl']).cumprod()
                daily_stats['asset_class'] = asset_class
                daily_stats['model'] = model_name
                
                # 保存为 CSV
                file_name = f'pnl_{asset_class.replace(" ", "_")}_{model_name.replace("(", "").replace(")", "")}.csv'
                pnl_path = output_dir / file_name
                daily_stats.to_csv(pnl_path, index=False)
                exported_files.append((file_name, pnl_path))
                print(f"  ✅ {asset_class:20s} - {model_name:10s}: {file_name}")
                pnl_count += 1
    
    print(f"\n✅ 已导出 {pnl_count} 个 PnL 数据文件")
    
    # 总结
    print("\n" + "=" * 80)
    print("📊 导出完成总结")
    print("=" * 80)
    print(f"✅ 共导出 {len(exported_files)} 个文件")
    
    if IN_COLAB:
        print(f"\n📂 所有文件已保存到 Google Drive:")
        print(f"   📍 {output_dir}")
        print(f"\n💡 如何获取文件:")
        print(f"   1. 打开 Google Drive")
        print(f"   2. 导航到: My Drive → IEOR4733_Project → reproduction_of_figures")
        print(f"   3. 下载所有 CSV 文件到本地")
        print(f"   4. 放到本地的: /Users/ladymie/Documents/GitHub/IEOR4733_Project/reproduction_of_figures/")
        print(f"\n📋 文件列表:")
        for fname, _ in exported_files:
            print(f"     • {fname}")
    else:
        print(f"\n✅ 所有文件已保存到:")
        print(f"   {output_dir}")
        print(f"\n📋 文件列表:")
        for fname, _ in exported_files:
            print(f"     • {fname}")

except Exception as e:
    print(f"❌ 导出失败: {e}")
    import traceback
    traceback.print_exc()

检查必要数据...
✅ my_table2_full                : 已定义
✅ my_table3_full                : 已定义
✅ pnl_by_model_by_ticker        : 已定义
✅ PAPER_TICKERS_BY_ASSET_CLASS  : 已定义

导出所有结果为 CSV 格式...
📍 Colab 环境 - 保存到 Google Drive
📂 输出目录: /content/drive/MyDrive/IEOR4733_Project/reproduction_of_figures

✅ Table 2 已保存: a2c_all_results_table2.csv
✅ Table 3 已保存: a2c_all_results_table3.csv
✅ Compare Table 2 已保存: a2c_compare_vs_paper_table2.csv
✅ Compare Table 3 已保存: a2c_compare_vs_paper_table3.csv

导出 PnL 数据...
  ✅ Commodity            - Long      : pnl_Commodity_Long.csv
  ✅ Commodity            - Sign(R)   : pnl_Commodity_SignR.csv
  ✅ Commodity            - MACD      : pnl_Commodity_MACD.csv
  ✅ Commodity            - A2C       : pnl_Commodity_A2C.csv
  ✅ Equity Index         - Long      : pnl_Equity_Index_Long.csv
  ✅ Equity Index         - Sign(R)   : pnl_Equity_Index_SignR.csv
  ✅ Equity Index         - MACD      : pnl_Equity_Index_MACD.csv
  ✅ Equity Index         - A2C       : pnl_Equity_Index_A2C.csv
